# WYN Distribution Product Scraper

This notebook scrapes all products from wyndistribution.com and exports them to a Shopify-compatible CSV file.

## Instructions
1. Click **Runtime** → **Run all** (or press Ctrl+F9)
2. Wait for the scraper to complete (may take 30-60 minutes depending on product count)
3. Download your files from the **Files** panel on the left, or use the download links at the end

---

## Step 1: Install Dependencies
This installs Chrome, ChromeDriver, and required Python packages.

In [ ]:
# Install Chrome and ChromeDriver for Colab
%%capture
!apt-get update
!apt-get install -y chromium-chromedriver
!pip install selenium requests

# Add chromedriver to path
import sys
sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

print("✅ Dependencies installed successfully!")

## Step 2: Configure Login Credentials
Your WYN Distribution login credentials are pre-filled below. Edit if needed.

In [ ]:
# Login credentials - edit these if needed
USERNAME = "joshua@oilslickpad.com"
PASSWORD = "710_Sl1ck"

print(f"✅ Using login: {USERNAME}")

## Step 3: Run the Scraper
This will:
1. Log into the website
2. Find all product categories
3. Scrape every product's details
4. Download all product images
5. Export to Shopify CSV format

In [ ]:
import os
import re
import csv
import time
import json
import requests
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, urlparse

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException
)

# Configuration
BASE_URL = "https://wyndistribution.com"
LOGIN_URL = f"{BASE_URL}/my-account/"
SHOP_URL = f"{BASE_URL}/shop/"

# Output directories
OUTPUT_DIR = Path("output")
IMAGES_DIR = OUTPUT_DIR / "images"
DATA_DIR = OUTPUT_DIR / "data"

# Create directories
OUTPUT_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print("✅ Configuration loaded")

In [ ]:
class WYNScraperColab:
    """Scraper for WYN Distribution wholesale website - Colab version."""

    def __init__(self, username, password):
        """Initialize the scraper."""
        self.username = username
        self.password = password
        self.driver = None
        self.session = requests.Session()
        self.products = []
        self.categories = []
        self.failed_products = []

    def setup_driver(self):
        """Configure Chrome for Colab environment."""
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        
        # Colab-specific: use system chromium
        chrome_options.binary_location = "/usr/bin/chromium-browser"
        
        service = Service("/usr/bin/chromedriver")
        self.driver = webdriver.Chrome(service=service, options=chrome_options)
        self.driver.implicitly_wait(10)
        print("✅ Chrome browser initialized")

    def login(self):
        """Log into the WYN Distribution website."""
        print("🔐 Logging in...")
        self.driver.get(LOGIN_URL)
        time.sleep(3)

        try:
            # Wait for login form
            username_field = WebDriverWait(self.driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='username'], input#username"))
            )
            password_field = self.driver.find_element(By.CSS_SELECTOR, "input[name='password'], input#password")

            # Enter credentials
            username_field.clear()
            username_field.send_keys(self.username)
            password_field.clear()
            password_field.send_keys(self.password)

            # Click login
            login_button = self.driver.find_element(By.CSS_SELECTOR, "button[name='login'], input[name='login'], button[type='submit']")
            login_button.click()

            time.sleep(4)

            # Verify login
            WebDriverWait(self.driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".woocommerce-MyAccount-content, a[href*='logout'], .logged-in"))
            )

            print("✅ Successfully logged in!")

            # Transfer cookies to requests session
            for cookie in self.driver.get_cookies():
                self.session.cookies.set(cookie['name'], cookie['value'])

            return True

        except TimeoutException:
            print("❌ Login failed - timeout")
            self.driver.save_screenshot(str(OUTPUT_DIR / "login_failed.png"))
            return False
        except NoSuchElementException as e:
            print(f"❌ Login failed - element not found: {e}")
            return False

    def get_categories(self):
        """Extract all product categories."""
        print("📂 Fetching product categories...")
        self.driver.get(SHOP_URL)
        time.sleep(3)

        categories = []

        try:
            selectors = [
                "ul.product-categories a",
                ".widget_product_categories a",
                ".product-category a",
                "a[href*='/product-category/']"
            ]

            for selector in selectors:
                try:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    for elem in elements:
                        href = elem.get_attribute("href")
                        name = elem.text.strip()
                        if href and "/product-category/" in href and name:
                            categories.append({"name": name, "url": href})
                except:
                    continue

            # Remove duplicates
            seen = set()
            unique_categories = []
            for cat in categories:
                if cat["url"] not in seen:
                    seen.add(cat["url"])
                    unique_categories.append(cat)

            self.categories = unique_categories
            print(f"✅ Found {len(self.categories)} categories")

            if not self.categories:
                self.categories = [{"name": "All Products", "url": SHOP_URL}]

            return self.categories

        except Exception as e:
            print(f"⚠️ Error fetching categories: {e}")
            self.categories = [{"name": "All Products", "url": SHOP_URL}]
            return self.categories

    def get_all_product_urls(self):
        """Get all product URLs from all pages."""
        product_urls = set()
        pages_to_scrape = [SHOP_URL] + [cat["url"] for cat in self.categories]

        for page_url in pages_to_scrape:
            print(f"📄 Scanning: {page_url[:60]}...")
            urls = self._get_products_from_listing(page_url)
            product_urls.update(urls)

        print(f"✅ Found {len(product_urls)} unique products")
        return list(product_urls)

    def _get_products_from_listing(self, listing_url):
        """Get product URLs from a listing page with pagination."""
        product_urls = set()
        current_url = listing_url
        page_num = 1

        while current_url:
            self.driver.get(current_url)
            time.sleep(2)

            try:
                product_selectors = [
                    "a.woocommerce-LoopProduct-link",
                    ".products .product a[href*='/product/']",
                    "ul.products li.product a",
                    "a[href*='/product/']"
                ]

                for selector in product_selectors:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    for elem in elements:
                        href = elem.get_attribute("href")
                        if href and "/product/" in href:
                            product_urls.add(href)
                    if elements:
                        break
            except Exception as e:
                pass

            # Check for next page
            current_url = None
            try:
                next_selectors = ["a.next.page-numbers", ".woocommerce-pagination a.next", "a[rel='next']"]
                for selector in next_selectors:
                    try:
                        next_link = self.driver.find_element(By.CSS_SELECTOR, selector)
                        current_url = next_link.get_attribute("href")
                        page_num += 1
                        break
                    except NoSuchElementException:
                        continue
            except:
                pass

        return product_urls

    def scrape_product(self, product_url):
        """Scrape a single product page."""
        try:
            self.driver.get(product_url)
            time.sleep(2)

            product = {
                "url": product_url,
                "handle": "",
                "title": "",
                "body_html": "",
                "vendor": "",
                "product_type": "",
                "tags": "",
                "variant_sku": "",
                "variant_price": "",
                "variant_compare_at_price": "",
                "image_src": "",
                "image_alt_text": "",
                "additional_images": [],
                "categories": [],
                "variations": [],
                "meta_title": "",
                "meta_description": ""
            }

            # Title
            try:
                title_elem = self.driver.find_element(By.CSS_SELECTOR, "h1.product_title, .product-title h1, h1.entry-title")
                product["title"] = title_elem.text.strip()
                product["handle"] = self._generate_handle(product["title"])
            except NoSuchElementException:
                pass

            # Price
            try:
                for selector in [".price .woocommerce-Price-amount bdi", ".price .amount", "p.price span.woocommerce-Price-amount"]:
                    try:
                        price_elem = self.driver.find_element(By.CSS_SELECTOR, selector)
                        product["variant_price"] = self._clean_price(price_elem.text.strip())
                        break
                    except NoSuchElementException:
                        continue
                try:
                    regular_price = self.driver.find_element(By.CSS_SELECTOR, ".price del .amount")
                    product["variant_compare_at_price"] = self._clean_price(regular_price.text.strip())
                except NoSuchElementException:
                    pass
            except:
                pass

            # SKU
            try:
                for selector in [".sku_wrapper .sku", ".product_meta .sku", "span.sku"]:
                    try:
                        sku_elem = self.driver.find_element(By.CSS_SELECTOR, selector)
                        product["variant_sku"] = sku_elem.text.strip()
                        break
                    except NoSuchElementException:
                        continue
            except:
                pass

            # Description
            try:
                for selector in [".woocommerce-product-details__short-description", "#tab-description"]:
                    try:
                        desc_elem = self.driver.find_element(By.CSS_SELECTOR, selector)
                        product["body_html"] = desc_elem.get_attribute("innerHTML").strip()
                        break
                    except NoSuchElementException:
                        continue
            except:
                pass

            # Categories
            try:
                cat_elems = self.driver.find_elements(By.CSS_SELECTOR, ".posted_in a")
                product["categories"] = [elem.text.strip() for elem in cat_elems]
                product["product_type"] = product["categories"][0] if product["categories"] else ""
                product["tags"] = ", ".join(product["categories"])
            except:
                pass

            # Vendor/Brand
            try:
                for selector in [".tagged_as a", "a[href*='/brand/']"]:
                    try:
                        brand_elem = self.driver.find_element(By.CSS_SELECTOR, selector)
                        product["vendor"] = brand_elem.text.strip()
                        break
                    except NoSuchElementException:
                        continue
            except:
                pass

            # Main Image
            try:
                for selector in [".woocommerce-product-gallery__image img", ".wp-post-image"]:
                    try:
                        img_elem = self.driver.find_element(By.CSS_SELECTOR, selector)
                        img_src = img_elem.get_attribute("data-large_image") or img_elem.get_attribute("src")
                        if img_src:
                            product["image_src"] = img_src
                            product["image_alt_text"] = img_elem.get_attribute("alt") or product["title"]
                            break
                    except NoSuchElementException:
                        continue
            except:
                pass

            # Additional Images
            try:
                gallery_imgs = self.driver.find_elements(By.CSS_SELECTOR, ".woocommerce-product-gallery__image:not(:first-child) img")
                for img in gallery_imgs:
                    img_src = img.get_attribute("data-large_image") or img.get_attribute("src")
                    if img_src and img_src != product["image_src"]:
                        product["additional_images"].append(img_src)
            except:
                pass

            # Variations
            try:
                variation_form = self.driver.find_element(By.CSS_SELECTOR, "form.variations_form")
                variations_data = variation_form.get_attribute("data-product_variations")
                if variations_data:
                    product["variations"] = json.loads(variations_data)
            except:
                product["variations"] = []

            # Meta
            try:
                meta_title = self.driver.find_element(By.CSS_SELECTOR, "meta[property='og:title']")
                product["meta_title"] = meta_title.get_attribute("content")
            except:
                product["meta_title"] = product["title"]

            self.products.append(product)
            return product

        except Exception as e:
            self.failed_products.append({"url": product_url, "error": str(e)})
            return None

    def download_image(self, image_url, product_handle, index=0):
        """Download an image."""
        if not image_url:
            return None
        try:
            ext = os.path.splitext(urlparse(image_url).path)[1] or ".jpg"
            filename = f"{product_handle}_{index}{ext}"
            filepath = IMAGES_DIR / filename
            if filepath.exists():
                return str(filepath)
            response = self.session.get(image_url, timeout=30)
            response.raise_for_status()
            with open(filepath, 'wb') as f:
                f.write(response.content)
            return str(filepath)
        except Exception as e:
            return None

    def download_all_images(self):
        """Download all product images."""
        print("📸 Downloading images...")
        total = len(self.products)
        for i, product in enumerate(self.products, 1):
            handle = product.get("handle", "unknown")
            if product.get("image_src"):
                self.download_image(product["image_src"], handle, 0)
            for j, img_url in enumerate(product.get("additional_images", []), 1):
                self.download_image(img_url, handle, j)
            if i % 20 == 0:
                print(f"  Downloaded images for {i}/{total} products...")
            time.sleep(0.3)
        print("✅ Images downloaded")

    def export_to_shopify_csv(self, filename="shopify_products.csv"):
        """Export to Shopify CSV format."""
        filepath = DATA_DIR / filename
        headers = [
            "Handle", "Title", "Body (HTML)", "Vendor", "Product Category", "Type", "Tags",
            "Published", "Option1 Name", "Option1 Value", "Option2 Name", "Option2 Value",
            "Option3 Name", "Option3 Value", "Variant SKU", "Variant Grams",
            "Variant Inventory Tracker", "Variant Inventory Qty", "Variant Inventory Policy",
            "Variant Fulfillment Service", "Variant Price", "Variant Compare At Price",
            "Variant Requires Shipping", "Variant Taxable", "Variant Barcode",
            "Image Src", "Image Position", "Image Alt Text", "Gift Card",
            "SEO Title", "SEO Description", "Variant Weight Unit", "Status"
        ]
        rows = []
        for product in self.products:
            handle = product.get("handle", "")
            variations = product.get("variations", [])
            if variations:
                for i, var in enumerate(variations):
                    row = self._create_shopify_row(product, variation=var, is_first=(i == 0))
                    rows.append(row)
            else:
                row = self._create_shopify_row(product)
                rows.append(row)
            for img_idx, img_url in enumerate(product.get("additional_images", []), 2):
                img_row = {key: "" for key in headers}
                img_row["Handle"] = handle
                img_row["Image Src"] = img_url
                img_row["Image Position"] = img_idx
                img_row["Image Alt Text"] = product.get("title", "")
                rows.append(img_row)
        with open(filepath, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=headers, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(rows)
        print(f"✅ Exported {len(self.products)} products to {filepath}")
        return filepath

    def _create_shopify_row(self, product, variation=None, is_first=True):
        """Create a Shopify CSV row."""
        row = {
            "Handle": product.get("handle", ""),
            "Title": product.get("title", "") if is_first else "",
            "Body (HTML)": product.get("body_html", "") if is_first else "",
            "Vendor": product.get("vendor", "") if is_first else "",
            "Product Category": product.get("product_type", "") if is_first else "",
            "Type": product.get("product_type", "") if is_first else "",
            "Tags": product.get("tags", "") if is_first else "",
            "Published": "TRUE" if is_first else "",
            "Option1 Name": "", "Option1 Value": "",
            "Option2 Name": "", "Option2 Value": "",
            "Option3 Name": "", "Option3 Value": "",
            "Variant SKU": product.get("variant_sku", ""),
            "Variant Grams": "",
            "Variant Inventory Tracker": "shopify",
            "Variant Inventory Qty": "",
            "Variant Inventory Policy": "deny",
            "Variant Fulfillment Service": "manual",
            "Variant Price": product.get("variant_price", ""),
            "Variant Compare At Price": product.get("variant_compare_at_price", ""),
            "Variant Requires Shipping": "TRUE",
            "Variant Taxable": "TRUE",
            "Variant Barcode": "",
            "Image Src": product.get("image_src", "") if is_first else "",
            "Image Position": "1" if is_first and product.get("image_src") else "",
            "Image Alt Text": product.get("image_alt_text", "") if is_first else "",
            "Gift Card": "FALSE" if is_first else "",
            "SEO Title": product.get("meta_title", "") if is_first else "",
            "SEO Description": product.get("meta_description", "") if is_first else "",
            "Variant Weight Unit": "lb",
            "Status": "active" if is_first else ""
        }
        if variation:
            attrs = variation.get("attributes", {})
            option_num = 1
            for attr_name, attr_value in attrs.items():
                if option_num <= 3:
                    row[f"Option{option_num} Name"] = attr_name.replace("attribute_pa_", "").replace("attribute_", "").replace("-", " ").title()
                    row[f"Option{option_num} Value"] = attr_value
                    option_num += 1
            row["Variant SKU"] = variation.get("sku", "")
            row["Variant Price"] = self._clean_price(str(variation.get("display_price", "")))
            if variation.get("display_regular_price"):
                row["Variant Compare At Price"] = self._clean_price(str(variation.get("display_regular_price", "")))
        return row

    def export_to_json(self, filename="products.json"):
        """Export to JSON format."""
        filepath = DATA_DIR / filename
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(self.products, f, indent=2, ensure_ascii=False)
        print(f"✅ Exported to {filepath}")
        return filepath

    @staticmethod
    def _generate_handle(title):
        if not title:
            return ""
        handle = title.lower()
        handle = re.sub(r'[^a-z0-9\s-]', '', handle)
        handle = re.sub(r'[\s_]+', '-', handle)
        handle = re.sub(r'-+', '-', handle)
        return handle.strip('-')[:255]

    @staticmethod
    def _clean_price(price_str):
        if not price_str:
            return ""
        cleaned = re.sub(r'[^\d.]', '', str(price_str))
        try:
            return str(float(cleaned))
        except ValueError:
            return ""

    def close(self):
        if self.driver:
            self.driver.quit()

print("✅ Scraper class loaded")

In [ ]:
# Run the scraper
print("="*50)
print("🚀 Starting WYN Distribution Scraper")
print("="*50)

start_time = datetime.now()
scraper = WYNScraperColab(USERNAME, PASSWORD)

try:
    # Setup browser
    scraper.setup_driver()
    
    # Login
    if not scraper.login():
        raise Exception("Login failed - check credentials")
    
    # Get categories
    scraper.get_categories()
    
    # Get all product URLs
    product_urls = scraper.get_all_product_urls()
    
    if not product_urls:
        raise Exception("No products found")
    
    # Scrape each product
    print(f"\n🛒 Scraping {len(product_urls)} products...")
    for i, url in enumerate(product_urls, 1):
        scraper.scrape_product(url)
        if i % 10 == 0:
            print(f"  Progress: {i}/{len(product_urls)} products scraped")
        time.sleep(1)  # Rate limiting
    
    # Download images
    scraper.download_all_images()
    
    # Export data
    scraper.export_to_shopify_csv()
    scraper.export_to_json()
    
    # Summary
    elapsed = datetime.now() - start_time
    print("\n" + "="*50)
    print("🎉 SCRAPING COMPLETE!")
    print("="*50)
    print(f"✅ Total products scraped: {len(scraper.products)}")
    print(f"❌ Failed products: {len(scraper.failed_products)}")
    print(f"⏱️ Time elapsed: {elapsed}")
    print("="*50)

except Exception as e:
    print(f"\n❌ Error: {e}")
    
finally:
    scraper.close()

## Step 4: Download Your Files

Run the cell below to create download links for your files.

In [ ]:
from google.colab import files
import shutil

print("📁 Your files are ready!\n")

# List output files
print("Files created:")
for f in DATA_DIR.glob("*"):
    size = f.stat().st_size / 1024
    print(f"  📄 {f.name} ({size:.1f} KB)")

img_count = len(list(IMAGES_DIR.glob("*")))
print(f"  🖼️ {img_count} images in /output/images/")

print("\n" + "="*50)
print("DOWNLOAD OPTIONS:")
print("="*50)

# Create zip of all files
print("\n📦 Creating zip file of all data...")
shutil.make_archive("wyn_products", 'zip', OUTPUT_DIR)
print("✅ Created wyn_products.zip")

print("\n👇 Click the links below to download:\n")

In [ ]:
# Download Shopify CSV (main file you need)
print("📥 Downloading Shopify CSV...")
files.download('output/data/shopify_products.csv')

In [ ]:
# Download complete zip with all data and images
print("📥 Downloading complete zip (includes images)...")
files.download('wyn_products.zip')

---

## How to Import into Shopify

1. Go to **Shopify Admin** → **Products** → **Import**
2. Click **Add file** and upload `shopify_products.csv`
3. Review the import preview
4. Click **Import products**

### Notes:
- Images are linked by URL from the original site
- If you need local images, extract them from `wyn_products.zip`
- Products with variations will have multiple rows (one per variant)

---

## Troubleshooting

**Login failed?**
- Double-check username/password in Step 2
- The website may have changed - check if you can log in manually

**No products found?**
- The site structure may have changed
- Check for CAPTCHA or bot detection

**Timeout errors?**
- The site may be slow - try running again
- Colab sessions have time limits - for very large catalogs, you may need to run in batches